## Module 6-1 Statistical Testing and Regression

### 1. t-tests

In [ ]:
from scipy import stats
import numpy as np

#### 1.1. One-sample t-test

Used to test whether the mean of a sample differs from a known value (e.g., population mean μ₀).

In [ ]:
# Example data
data = np.array([5.1, 5.3, 4.9, 5.0, 5.2])

# Hypothesized population mean
mu_0 = 5.0

# One-sample t-test
t_stat, p_val = stats.ttest_1samp(data, mu_0)

print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")

#### 1.2. Independent two-sample t-test

Used to compare the means of two independent groups (e.g., control vs. treatment).

In [ ]:
group1 = np.array([4.9, 5.1, 5.0, 4.8, 5.2])
group2 = np.array([5.4, 5.6, 5.5, 5.7, 5.8, 7.0, 3.2])

# Equal variance assumed (default)
t_stat, p_val = stats.ttest_ind(group1, group2, equal_var=True)

print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")

#### 1.3. Paired (dependent) t-test

Used when comparing two related samples (e.g., before vs. after measurements on the same subjects).

In [ ]:
before = np.array([10, 12, 13, 12, 11])
after  = np.array([11, 14, 13, 13, 12])

# Paired t-test
t_stat, p_val = stats.ttest_rel(before, after)

print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.3f}")

### 2. Linear Regressions

We will reuse the data produced from Section 3-1e: `../data/Smirk_FirmYear.csv`. Recall the variables in that data are as follows:

| Paper's variable | Our column | Source |
| --- | --- | --- |
| $IV\_SKEW$ (DV) | `iv_skew` | OptionMetrics | 
| $ATM\_IV$ | `atm_iv` | OptionMetrics | 
| $FIRM\_SIZE$ | `firm_size` | Compustat | 
| $LEVERAGE$ | `leverage` | Compustat | 
| $MB$ | `mb` | Compustat |

Specifications:

1. Standard errors clustered by firm and year.
2. Firm and year fixed effects.

In [ ]:
import numpy as np
import pandas as pd
import pyfixest as pf

#### 2.1. Building the estimation sample

In [ ]:
smirk = pd.read_csv(
    "../data/Smirk_FirmYear.csv",
    dtype={"gvkey": str, "fyear": int}
)
print(f"{len(smirk):,} firm-years from the Module 3 exercise")
smirk.head()

##### Drop missing values

In [ ]:
smirk.isna().sum()

In [ ]:
reg_sample = smirk.dropna()

##### Winsorize

In [ ]:
for var in ["Size", "Leverage", 'MB', 'ATM_IV', 'IV_Skew']:
    reg_sample[f"{var}_w"] = reg_sample[var].clip(reg_sample[var].quantile(0.01), reg_sample[var].quantile(0.99))

In [ ]:
reg_sample.columns

In [ ]:
reg_sample[['Size', 'Size_w',  'Leverage', 'Leverage_w', 'MB', 'MB_w', 'ATM_IV', 'ATM_IV_w', 'IV_Skew','IV_Skew_w']].describe().T

#### 2.2. Regression with `pyfixest`

[`pyfixest`](https://github.com/py-econometrics/pyfixest) is the closest package in the Python that mimics to Stata's `reghdfe`. One function.

`pf.feols()` can: run OLS, absorbed fixed effects, and clustered standard errors at two dimensions, with the degrees-of-freedom bookkeeping handled for you.

The model is written as an **R-style formula string**:

```R
depvar ~ x1 + x2 + x3 + ... | fe1 + fe2 + fe...
```

(leave the `|` off and you get plain OLS with an intercept) 

Other useful pieces of the syntax:

- `C(fyear)` for a categorical (like `i.fyear` in Stata)
- `x1:x2` for an interaction alone
- `x1*x2` for both main effects and their interaction
- `i(group, x)` for a full set of interacted dummies
- `vcov` argument chooses the standard errors

In [ ]:
ols = pf.feols("IV_Skew_w ~ ATM_IV_w + Size_w + Leverage_w + MB_w | gvkey + fyear",
               data=reg_sample)
ols.summary()

In [ ]:
ols.tidy()

In [ ]:
# `.coefplot()` draws the coefficients with their confidence intervals.
ols.coefplot()

#### 2.3. Standard errors: clustering by two dimensions

The standard errors are not IID in our regression above. A firm's risk is correlated with its own risk in adjacent years, and every firm's risk moves together when the whole market gets nervous.

Thus, the residuals are correlated in two directions at once:

- **Within firm, across years** (a *time-series* dependence): firm characteristics we have not modelled persist.
- **Within year, across firms** (a *cross-sectional* dependence): common shocks hit every firm together (e.g., COVID).

Petersen ([2009](https://doi.org/10.1093/rfs/hhn053)) showed that in accounting and finance panels this routinely overstates *t*-statistics by a factor of two or more, and two-way clustering is now very common in the literature.

In `pyfixest` we add `vcov` argument: `{"CRV1": "gvkey + fyear"}`, where CRV1 is the standard cluster-robust variance estimator and the two variables after the colon are the clustering dimensions.

In [ ]:
ols_cl = pf.feols("IV_Skew_w ~ ATM_IV_w + Size_w + Leverage_w + MB_w | gvkey + fyear",
                    data=reg_sample, 
                    vcov={"CRV1": "gvkey + fyear"})
ols_cl.summary()

In [ ]:
iid, clustered = ols.tidy(), ols_cl.tidy()

pd.DataFrame({
    "coef": iid["Estimate"],
    "se (iid)": iid["Std. Error"],
    "se (firm & year)": clustered["Std. Error"],
    "inflation": clustered["Std. Error"] / iid["Std. Error"],
    "t (iid)": iid["t value"],
    "t (firm & year)": clustered["t value"],
}).round(4)

#### 2.4. Output a regression table

`pf.etable()` help produce Journal tables, which put the specifications side by side, coefficients with standard errors beneath them in parentheses, stars for the conventional significance levels, and a block at the bottom recording which fixed effects were included.

- `labels` renames variables to the paper's notation
- `felabels` names the fixed-effect rows
- `file_name="table.tex"` writes it straight to LaTeX.

In [ ]:
LABELS = {
    "IV_Skew_w": "IV_SKEW", 
    "ATM_IV_w": "ATM_IV", 
    "Size_w": "FIRM_SIZE",
    "Leverage_w": "LEVERAGE", 
    "MB_w": "MB"}

pf.etable(
    [ols_cl],
    labels=LABELS,
    felabels={"fyear": "Year fixed effects", "gvkey": "Firm fixed effects"},
    model_heads=["FE Regression with two-way clustering"],
    signif_code=[0.01, 0.05, 0.10],
    coef_fmt="b:.3f* \n (t:.2f)",
    notes="""
    t-values are shown in parentheses.
    Standard errors clustered by firm and year. 
    *** p<0.01, ** p<0.05, * p<0.10.
          """
)

We can add more columns to the table:

In [ ]:
# Column (1): no fixed effects
no_fe = pf.feols("IV_Skew_w ~ ATM_IV_w + Size_w + Leverage_w + MB_w",
                data=reg_sample, 
                vcov={"CRV1": "gvkey + fyear"})
no_fe.summary()

In [ ]:
# Column (2): only year fixed effect
year_fe = pf.feols("IV_Skew_w ~ ATM_IV_w + Size_w + Leverage_w + MB_w | gvkey",
                    data=reg_sample, 
                    vcov={"CRV1": "gvkey + fyear"})
year_fe.summary()

In [ ]:
out_table = pf.etable(
    [no_fe, year_fe, ols_cl],
    labels=LABELS,
    felabels={"fyear": "Year fixed effects", "gvkey + fyear": "Firm and Year fixed effects"},
    model_heads=["No FE", "Year FE", "Year and Firm FE"],
    signif_code=[0.01, 0.05, 0.10],
    coef_fmt="b:.3f* \n (t:.2f)",
    type = "df", # Output as a Pandas.DataFrame
    notes="""
    t-values are shown in parentheses.
    Standard errors clustered by firm and year. 
    *** p<0.01, ** p<0.05, * p<0.10.
    """
)

In [ ]:
out_table.to_csv("regression_results.csv", index=False)

### 3. Logistic Regressions

Similar syntax using `pf.feglm`:

```py
logit = pf.feglm(
    "y ~ x1 + x2 + x3 + ... | gvkey + fyear",
    data=df,
    family="logit",
    vcov={"CRV1": "gvkey + fyear"}
)
```

In addition, you can use `marginaleffects`, so you can obtain average marginal effects rather than interpreting log-odds coefficients directly:

```py
from marginaleffects import avg_slopes
avg_slopes(fit, variables=["x1", "x2"])
```